In [138]:
!git clone https://github.com/cpoiato/HYDROSIGHT.git /content/HYDROSIGHT

fatal: destination path '/content/HYDROSIGHT' already exists and is not an empty directory.


In [139]:
%cd /content/HYDROSIGHT
!git pull --ff-only

/content/HYDROSIGHT
Already up to date.


In [140]:
%cd /content/HYDROSIGHT/src/hydrosight/eda/notebooks

/content/HYDROSIGHT/src/hydrosight/eda/notebooks


In [141]:
from pathlib import Path

repo_path = Path("/content/HYDROSIGHT")

print("Repository folder exists:", repo_path.exists())
print("Git metadata exists:", (repo_path / ".git").exists())
print("Current directory:", Path.cwd())

Repository folder exists: True
Git metadata exists: True
Current directory: /content/HYDROSIGHT/src/hydrosight/eda/notebooks


In [142]:
from google.colab import drive
import json

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [143]:
from pathlib import Path
import sys

ROOT_DIR = Path("/content/HYDROSIGHT")
sys.path.insert(0, str(ROOT_DIR / "src"))
RAW_DIR = ROOT_DIR / "data" / "raw" / "schools"

assert RAW_DIR.exists(), f"Folder not found: {RAW_DIR}"
assert RAW_DIR.exists(), f"Folder not found: {RAW_DIR}"

print("Project:", ROOT_DIR)
print("Raw data:", RAW_DIR)

print("\nAvailable pipe files:")
for file in sorted(RAW_DIR.glob("*.geojson")):
    print(file.name)

Project: /content/HYDROSIGHT
Raw data: /content/HYDROSIGHT/data/raw/schools

Available pipe files:
schools_20261024T153107Z.geojson


In [144]:
import pandas as pd
from IPython.display import display


files = sorted(RAW_DIR.glob("schools_*.geojson"))

assert files, "No schools snapshots found."

latest_file = files[-1]

df = pd.read_json(latest_file)
df.shape

(506, 2)

In [145]:
display("Dataset Information")
df.info()

'Dataset Information'

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   type      506 non-null    object
 1   features  506 non-null    object
dtypes: object(2)
memory usage: 8.0+ KB


In [146]:
display("Sample Data")
display(df)

'Sample Data'

,type,features
0,FeatureCollection,"{'type': 'Feature', 'id': 355960, 'geometry': ..."
1,FeatureCollection,"{'type': 'Feature', 'id': 355961, 'geometry': ..."
2,FeatureCollection,"{'type': 'Feature', 'id': 355962, 'geometry': ..."
3,FeatureCollection,"{'type': 'Feature', 'id': 355963, 'geometry': ..."
4,FeatureCollection,"{'type': 'Feature', 'id': 355964, 'geometry': ..."
...,...,...
501,FeatureCollection,"{'type': 'Feature', 'id': 356520, 'geometry': ..."
502,FeatureCollection,"{'type': 'Feature', 'id': 356521, 'geometry': ..."
503,FeatureCollection,"{'type': 'Feature', 'id': 356522, 'geometry': ..."
504,FeatureCollection,"{'type': 'Feature', 'id': 356523, 'geometry': ..."


In [147]:
geometry = df["features"].iloc[0]

if isinstance(geometry, dict):
        print("Key:", geometry.keys())
        print("Type:", geometry.get("type"))
        print("Coordinates:", geometry.get("geometry"))
else:
        print("Type:", geometry.get("type"))
        print("Coordinates:", geometry.get("geometry"))

Key: dict_keys(['type', 'id', 'geometry', 'properties'])
Type: Feature
Coordinates: {'type': 'Point', 'coordinates': [-114.066353339901, 50.9224441230449]}


In [148]:
import geopandas as gpd

# Load Calgary city boundary from the official ArcGIS service
boundary_url = (
    "https://services1.arcgis.com/AVP60cs0Q9PEA8rH/"
    "arcgis/rest/services/City_Boundary/FeatureServer/0/query"
    "?where=1%3D1"
    "&outFields=*"
    "&outSR=4326"
    "&returnGeometry=true"
    "&f=geojson"
)

calgary_boundary = gpd.read_file(boundary_url)

# Validate the boundary geometry
print("Total records:", len(calgary_boundary))
print("CRS:", calgary_boundary.crs)
print("Geometry types:")
print(calgary_boundary.geometry.geom_type.value_counts())
print("Bounds:", calgary_boundary.total_bounds)
print("Valid geometries:", calgary_boundary.geometry.is_valid.sum())

assert (
    not calgary_boundary.empty
    and calgary_boundary.geometry.notna().all()
    and calgary_boundary.geometry.is_valid.all()
    and calgary_boundary.geometry.geom_type.isin(
        ["Polygon", "MultiPolygon"]
    ).all()
), "Invalid Calgary boundary geometry"

Total records: 1
CRS: EPSG:4326
Geometry types:
Polygon    1
Name: count, dtype: int64
Bounds: [-114.31580385   50.84282878 -113.85991285   51.21243208]
Valid geometries: 1


In [149]:
from shapely.geometry import shape

# Convert GeoJSON MultiLineString to Shapely geometry
schools_gdf = gpd.GeoDataFrame(
    df.copy(),
    geometry=df["features"].apply(
        lambda value: shape(json.loads(value))
        if isinstance(value, str) else shape(value)
    ),
    crs="EPSG:4326"
)

In [150]:
# Align coordinate reference systems
schools = schools_gdf.to_crs(calgary_boundary.crs)

# Merge municipal boundary geometries
city_boundary = calgary_boundary.geometry.union_all()

# Classify pipe locations
schools["boundary_status"] = "outside"

schools.loc[
    schools.geometry.intersects(city_boundary),
    "boundary_status"
] = "partially_inside"

schools.loc[
    schools.geometry.covered_by(city_boundary),
    "boundary_status"
] = "inside"

# Display validation results
print("schools boundary validation results:")
print(schools["boundary_status"].value_counts())

schools boundary validation results:
boundary_status
inside    506
Name: count, dtype: int64


In [151]:
print("Total records:", len(schools))
print("CRS:", schools.crs)


Total records: 506
CRS: EPSG:4326


In [152]:
print("Geometry types:")
print(schools.geometry.geom_type.value_counts())


Geometry types:
Point    506
Name: count, dtype: int64


In [153]:
print("Bounds:", schools.total_bounds)
print("Valid geometries:", schools.geometry.is_valid.sum())


Bounds: [-114.25112544   50.85933539 -113.86775991   51.18695075]
Valid geometries: 506


In [154]:
print("Null values:")
print(schools.isnull().sum())

Null values:
type               0
features           0
geometry           0
boundary_status    0
dtype: int64


In [155]:
import pandas as pd

# Extract properties and IDs from each feature for duplicate analysis
features_list = []

for idx, row in schools.iterrows():
    feature = row["features"]

    if isinstance(feature, dict):
        properties = feature.get("properties", {}).copy()
        properties["feature_id"] = feature.get("id")
        properties["geometry_type"] = feature.get("geometry", {}).get("type")
        properties["coordinates"] = str(
            feature.get("geometry", {}).get("coordinates")
        )
        features_list.append(properties)

# Create an auxiliary DataFrame with extracted feature metadata
df_features = pd.DataFrame(features_list)

# Identify duplicates by unique feature ID
duplicates_by_id = df_features[
    df_features.duplicated(subset=["feature_id"], keep=False)
]

# Identify duplicates by the combination of school name and address
duplicates_by_name_address = df_features[
    df_features.duplicated(subset=["NAME", "ADDRESS"], keep=False)
]

print(f"Total records duplicated by feature ID: {len(duplicates_by_id)}")
print(
    f"Total records duplicated by name and address: "
    f"{len(duplicates_by_name_address)}"
)

if not duplicates_by_name_address.empty:
    print("\nDuplicate records found by name and address:")

    display(
        duplicates_by_name_address[
            ["feature_id", "NAME", "ADDRESS", "CITY", "coordinates"]
        ].head(10)
    )
else:
    print("\nNo duplicate records found by name and address.")

Total records duplicated by feature ID: 0
Total records duplicated by name and address: 3

Duplicate records found by name and address:


,feature_id,NAME,ADDRESS,CITY,coordinates
501,356520,Access Academy,35 NORMANDY DR SW,Calgary,"[-114.119053007962, 51.0192933231405]"
502,356521,Access Academy,35 NORMANDY DR SW,Calgary,"[-114.119053007962, 51.0192933231405]"
503,356522,Access Academy,35 NORMANDY DR SW,Calgary,"[-114.119053007962, 51.0192933231405]"


In [156]:
# Consolidate records sharing the same school name, address, and coordinates
duplicate_mask = df_features.duplicated(
    subset=["NAME", "ADDRESS", "coordinates"],
    keep="first"
)

schools_clean = schools.loc[~duplicate_mask].copy()

print("=== SCHOOL LOCATION CONSOLIDATION ===")
print("Original records:", len(schools))
print("Removed duplicate locations:", duplicate_mask.sum())
print("Remaining records:", len(schools_clean))

=== SCHOOL LOCATION CONSOLIDATION ===
Original records: 506
Removed duplicate locations: 2
Remaining records: 504


In [157]:
df_clean = schools_clean.copy()

In [161]:
from datetime import datetime, timezone
import json
from shapely.geometry import mapping

def save_processed_data(df):
    # Définir le répertoire et le fichier de sortie
    output_dir = ROOT_DIR / "data" / "processed" / "schools"
    output_dir.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    output_file = output_dir / f"schools_clean_{timestamp}.geojson"

    # Pour éviter les erreurs de récursion lors de la sérialisation,
    # nous convertissons manuellement le DataFrame en une structure de dictionnaire standard
    records = []
    for idx, row in df.iterrows():
        record = {}
        for col in df.columns:
            val = row[col]
            if col == 'geometry' and val is not None:
                # Convertir l'objet Shapely Geometry en dictionnaire GeoJSON
                record[col] = mapping(val)
            elif isinstance(val, dict) or isinstance(val, list):
                record[col] = val
            elif hasattr(val, 'item'): # types numpy
                record[col] = val.item()
            else:
                record[col] = str(val) if not isinstance(val, (int, float, bool, type(None))) else val
        records.append(record)

    # Sauvegarde sécurisée à l'aide de la bibliothèque standard json
    with open(output_file, 'w', encoding='utf-8') as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')

    # Téléchargement local du fichier généré
    from google.colab import files as colab_files
    colab_files.download(str(output_file))

    print(f"Cleaned dataset saved safely: {output_file}")
    print(f"Exported records: {len(df)}")

In [162]:
# Appel sécurisé de la fonction de sauvegarde mise à jour
save_processed_data(df_clean)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Cleaned dataset saved safely: /content/HYDROSIGHT/data/processed/schools/schools_clean_20261101T224320Z.geojson
Exported records: 504
